In [ ]:
from resources.imports import *

import torch
import torch.nn as nn

from resources.MLdata import DATA
from resources.MLfunc import EarlyStopping, MaskedFieldMSELoss
from resources.MLfunc import hOpt_model, hOpt_compare, hOpt_best_summary
from resources.MLmodels import *

In [ ]:
%load_ext autoreload
%autoreload 2

# Field Output Surrogate

## Graph Neural Network (GNN)

### DATA

In [ ]:
FIELD_CONFIG = {
    "components": ("U1", "U2"),
    "drop_frame0": True,
    "layout": "auto",
    # Optional per-mode overrides:
    # "UT": {"path": "MLdata/MULTI-UT-disNodes-field.npz"},
    # "FT": {"path": "MLdata/MULTI-FT-disNodes-field.npz"},
}

DAT_GNN = DATA(
    path=1,
    path_add="",
    load=True,
    load_split=False,
    split_frac=0.9,
    split_seed=42,
    range_split=(True, False),
    save_split=False,
    LAT="FCC",
    dis="disNodes",
    dN=0.2,
    d_data="in",
    mechMode="UT",
    nsims=None,
    model="GNN",
    output_kind="field",
    field_config=FIELD_CONFIG,
    scale=("symm", "inout"),
    reduce_dim=False,
    round_decimals=5,
    geom_feats=(True, True),
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
DAT = DAT_GNN
if DAT.UTmechTest:
    print("UT input/output shape:", DAT.UT_train_in.shape, DAT.UT_train_out.shape)
    print("UT field shape:", DAT.UT_field_shape, DAT.UT_field_components)
if DAT.FTmechTest:
    print("FT input/output shape:", DAT.FT_train_in.shape, DAT.FT_train_out.shape)
    print("FT field shape:", DAT.FT_field_shape, DAT.FT_field_components)

### HPO

In [ ]:
# HYPERPARAMETER OPTIMIZATION HERE.
# Keep pool='node' for field outputs so the model returns one displacement-history vector per node.

# GNN_FIELD_hOpt_model_space = {
#     "gcn": {"depth": [2, 3, 4], "width": [64, 128, 256], "pool": ["node"]},
#     "gat": {"depth": [1, 2, 3], "width": [32, 64, 128], "heads": [1, 2, 4], "pool": ["node"]},
# }
# GNN_FIELD_hOpt_train_space = {
#     "optimizer": ["adamw"],
#     "lr": {"type": "float", "low": 3e-5, "high": 1e-3, "log": True},
#     "batch": [2, 4, 8],
#     "n_epochs": {"type": "fixed", "value": 150},
# }
# studies_GNN_FIELD = hOpt_compare(
#     typs=["gcn", "gat"], data=DAT_GNN, n_trials_per_typ=20,
#     model_space=GNN_FIELD_hOpt_model_space,
#     loss_space={"family": ["mse"]}, train_space=GNN_FIELD_hOpt_train_space,
#     seed=42, device=device, save=True, save_best_model=True,
#     name="GNN_field_hOpt", n_jobs=1, show_progress_bar=True,
# )
# hOpt_best_summary(studies_GNN_FIELD)

### MODEL

In [ ]:
GNN_FIELD = MODEL(
    typ=DAT.model,
    model=GNN(
        in_size=DAT.UT_train_in.shape[-1] if DAT.UTmechTest else DAT.FT_train_in.shape[-1],
        h_size=[128, 128, 128],
        out_size=DAT.UT_train_out.shape[-1] if DAT.UTmechTest else DAT.FT_train_out.shape[-1],
        act="gelu",
        block="gat",
        norm="layer",
        dropout=0.1,
        att_dropout=0.1,
        head_norm="layer",
        head_dropout=0.05,
        bias=True,
        heads=2,
        pool="node",
    ).to(device),
    lossf=MaskedFieldMSELoss(reduction="mean"),
    opt=("adamw", 1e-5),
    batch=4,
    lr=3e-4,
    data=DAT,
    mechMode=DAT.mechMode,
    scheduler=("min", 0.5, 20, 1e-4),
    earlyStop=EarlyStopping(patience=50, min_delta=1e-5, verbose=True),
    w_init="auto",
    device=device,
    optTrial=None,
    scan_matches_on_init=True,
)

GNN_FIELD.summary()

In [ ]:
GNN_FIELD.train(n_epochs=250, verbose=10, plot=True)

In [ ]:
GNN_FIELD.save(path=None, name=None)

In [ ]:
GNN_FIELD.predict(test_dataloader=None, plot=False, diagnostics=True, diag_plot=True)

In [ ]:
if DAT.UTmechTest:
    GNN_FIELD.plot_diagnostics(mode="UT", split="test")
if DAT.FTmechTest:
    GNN_FIELD.plot_diagnostics(mode="FT", split="test")

## Transformer

### DATA

In [ ]:
DAT_TR = DATA(
    path=1,
    path_add="",
    load=True,
    load_split=False,
    split_frac=0.9,
    split_seed=42,
    range_split=(True, False),
    save_split=False,
    LAT="FCC",
    dis="disNodes",
    dN=0.2,
    d_data="in",
    mechMode="UT",
    nsims=None,
    model="TR",
    output_kind="field",
    field_config=FIELD_CONFIG,
    scale=("symm", "inout"),
    reduce_dim=False,
    round_decimals=5,
    geom_feats=(True, True),
)

DAT = DAT_TR
if DAT.UTmechTest:
    print("UT input/output shape:", DAT.UT_train_in.shape, DAT.UT_train_out.shape)
    print("UT field shape:", DAT.UT_field_shape, DAT.UT_field_components)
if DAT.FTmechTest:
    print("FT input/output shape:", DAT.FT_train_in.shape, DAT.FT_train_out.shape)
    print("FT field shape:", DAT.FT_field_shape, DAT.FT_field_components)

### HPO

In [ ]:
# HYPERPARAMETER OPTIMIZATION HERE.
# Keep pool='node' for field outputs.

### MODEL

In [ ]:
TR_FIELD = MODEL(
    typ=DAT.model,
    model=Transformer(
        in_size=DAT.UT_train_in.shape[-1] if DAT.UTmechTest else DAT.FT_train_in.shape[-1],
        seq_len=DAT.UT_train_in.shape[-2] if DAT.UTmechTest else DAT.FT_train_in.shape[-2],
        h_size=[128],
        out_size=DAT.UT_train_out.shape[-1] if DAT.UTmechTest else DAT.FT_train_out.shape[-1],
        d_model=128,
        n_heads=4,
        n_layers=3,
        act="gelu",
        block="mlp",
        norm="layer",
        dropout=0.1,
        head_norm="layer",
        head_dropout=0.05,
        pool="node",
        use_cls_token=True,
    ).to(device),
    lossf=MaskedFieldMSELoss(reduction="mean"),
    opt=("adamw", 1e-5),
    batch=4,
    lr=3e-4,
    data=DAT,
    mechMode=DAT.mechMode,
    scheduler=("min", 0.5, 20, 1e-4),
    earlyStop=EarlyStopping(patience=50, min_delta=1e-5, verbose=True),
    w_init="auto",
    device=device,
    optTrial=None,
    scan_matches_on_init=True,
)

TR_FIELD.summary()

In [ ]:
TR_FIELD.train(n_epochs=250, verbose=10, plot=True)

In [ ]:
TR_FIELD.save(path=None, name=None)

In [ ]:
TR_FIELD.predict(test_dataloader=None, plot=False, diagnostics=True, diag_plot=True)

# END